# Download images:

# Import the required libraries:

In [ ]:
import time
import shutil
import hashlib
import requests
import pandas as pd
import numpy as np

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

c:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\venv_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load the csv file:

In [2]:
DATA_DIR = Path(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data")

CSV_PATH   = DATA_DIR / "dataset.csv"
RAW_DIR    = DATA_DIR / "raw_images"
UNIQUE_DIR = DATA_DIR / "unique_raw_images"

RAW_DIR.mkdir(parents=True, exist_ok=True)
UNIQUE_DIR.mkdir(parents=True, exist_ok=True)

# dataset.csv illa-na, inge clear-a error kaattum
if not CSV_PATH.exists():
    raise FileNotFoundError("dataset.csv inge illa: " + str(CSV_PATH))

df = pd.read_csv(CSV_PATH)
print("Total rows:", len(df))
df.head()

Total rows: 6557


,image_id,image_url,label,gender,age_group
0,1,https://images.unsplash.com/photo-172422561835...,REAL,Female,18-25
1,2,https://images.unsplash.com/photo-172422561873...,REAL,Male,50+
2,3,https://images.unsplash.com/photo-172422561834...,REAL,Female,36-50
3,4,https://images.unsplash.com/photo-172422561855...,REAL,Female,26-35
4,5,https://images.unsplash.com/photo-172422561823...,REAL,Male,36-50


## Step 2: Download all images into 'raw_images'
- Image will be saved in this format `image_id.jpg` 
- 8 will be downloaded per time

In [ ]:
def download_image(i):
    image_id = df.loc[i, "image_id"]
    url = df.loc[i, "image_url"]
    path = RAW_DIR / f"{image_id}.jpg"

    if path.exists():                        # already downloaded
        return image_id, "skipped"

    for attempt in range(3):                 # try 3 times
        try:
            response = requests.get(url, timeout=20) #type: ignore
            if response.status_code == 200 and response.headers.get("content-type", "").startswith("image"):
                path.write_bytes(response.content)
                return image_id, "downloaded"
        except requests.RequestException:
            time.sleep(2)

    return image_id, "failed"


with ThreadPoolExecutor(max_workers=8) as executor:
    results = list(tqdm(executor.map(download_image, range(len(df))), total=len(df)))

failed_ids = [image_id for image_id, status in results if status == "failed"]
print("Images in raw_images:", len(list(RAW_DIR.glob("*.jpg"))))
print("Failed downloads    :", len(failed_ids))
pd.DataFrame({"image_id": failed_ids}).to_csv(DATA_DIR / "failed_downloads.csv", index=False)

100%|██████████| 6557/6557 [18:28<00:00,  5.91it/s]

Images in raw_images: 5557
Failed downloads    : 1000


## Step 3: Find duplicate images

In [4]:
seen = {}              # hash -> first image_id
duplicate_ids = []     # extra copies

for image_id in df["image_id"]:
    path = RAW_DIR / f"{image_id}.jpg"
    if not path.exists():                    # failed download-ai skip
        continue

    image = Image.open(path).convert("RGB")
    h = hashlib.md5(image.tobytes() + str(image.size).encode()).hexdigest()

    if h in seen:
        duplicate_ids.append(image_id)
    else:
        seen[h] = image_id

print("Total images     :", len(list(RAW_DIR.glob("*.jpg"))))
print("Duplicate images :", len(duplicate_ids))
print("Unique images    :", len(seen))

pd.DataFrame({"duplicate_image_id": duplicate_ids}).to_csv(DATA_DIR / "duplicate_ids.csv", index=False)

Total images     : 5557
Duplicate images : 453
Unique images    : 5104


## Step 4: Copy the unique images into 'unique_raw_images'

In [5]:
unique_ids = list(seen.values())

for image_id in unique_ids:
    shutil.copy(RAW_DIR / f"{image_id}.jpg", UNIQUE_DIR / f"{image_id}.jpg")

# unique images-ukku matching csv (notebook 2 idhai use pannum)
unique_df = df[df["image_id"].isin(unique_ids)]
unique_df.to_csv(DATA_DIR / "unique_dataset.csv", index=False)

print("raw_images        :", len(list(RAW_DIR.glob("*.jpg"))))
print("unique_raw_images :", len(list(UNIQUE_DIR.glob("*.jpg"))))
print(unique_df["label"].value_counts())

raw_images        : 5557
unique_raw_images : 5104
label
FAKE    2746
REAL    2358
Name: count, dtype: int64


## Step-5: Check whether any duplicate image is present:

In [ ]:

def small_hash(path):
    image = Image.open(path)
    image.draft("L", (64, 64))                 # periya jpg-ai vegama open panna
    image = image.convert("L").resize((9, 8))  # grey, 9 x 8 pixels
    pixels = np.array(image, dtype=int)
    return (pixels[:, 1:] > pixels[:, :-1]).flatten()   # left pixel vida right pixel prakasama? -> 64 True/False

real_ids = list(unique_df[unique_df["label"] == "REAL"]["image_id"])
hashes = np.array([small_hash(RAW_DIR / f"{i}.jpg") for i in tqdm(real_ids)])

MAX_DIFF = 5          # 64 bits-la ethanai vithiyaasam varaikkum "similar" nu eduthukkalaam
pairs = []
for i in range(len(real_ids)):
    diff = (hashes[i + 1:] != hashes[i]).sum(axis=1)       # i-oda matha ellaa images-oda vithiyaasam
    for j in np.where(diff <= MAX_DIFF)[0]:
        pairs.append((real_ids[i], real_ids[i + 1 + j], int(diff[j])))

print("Real images checked      :", len(real_ids))
print("Near-duplicate pairs     :", len(pairs))
pairs_df = pd.DataFrame(pairs, columns=["image_id_1", "image_id_2", "difference"])
pairs_df.to_csv(DATA_DIR / "near_duplicate_pairs.csv", index=False)
pairs_df.head(10)

100%|██████████| 2358/2358 [00:34<00:00, 67.73it/s] 


Real images checked      : 2358
Near-duplicate pairs     : 27


,image_id_1,image_id_2,difference
0,82,1737,2
1,124,1666,0
2,144,917,5
3,144,1306,4
4,144,1667,0
5,160,2753,2
6,528,1646,3
7,615,624,0
8,675,926,4
9,747,1028,1


## Removing duplicate images in Unique image folder:

In [12]:
REMOVE_IF_DIFF = 2

to_remove = set(pairs_df[pairs_df["difference"] <= REMOVE_IF_DIFF]["image_id_2"])
print("Remove images:", len(to_remove))

for image_id in to_remove:
    path = UNIQUE_DIR / f"{image_id}.jpg"
    if path.exists():
        path.unlink()                          # unique_raw_images la irundhu mattum delete

unique_df = unique_df[~unique_df["image_id"].isin(to_remove)]
unique_df.to_csv(DATA_DIR / "unique_dataset.csv", index=False)

print("raw_images        :", len(list(RAW_DIR.glob("*.jpg"))))
print("unique_raw_images :", len(list(UNIQUE_DIR.glob("*.jpg"))))
print(unique_df["label"].value_counts())

Remove images: 16
raw_images        : 5557
unique_raw_images : 5088
label
FAKE    2746
REAL    2342
Name: count, dtype: int64
